# MedBoard — Train EfficientNet-B0 Classification Model

**Run this notebook either locally or in Google Colab to train the tumor classifier.**

### What this notebook does:
1. **Environment Setup**: Auto-detects Colab vs. Local VS Code; sets paths and dependencies.
2. **Dataset Loading**: Loads BRISC 2025 images for 4 classes (`glioma`, `meningioma`, `no_tumor`, `pituitary`) with stratified train/validation split (85%/15%) and data augmentations.
3. **Model Initialization**: Instantiates an ImageNet-pretrained **EfficientNet-B0** (~4.01M parameters).
4. **Training**: Trains with AdamW, Cosine Annealing LR, FP16 mixed precision, and early stopping.
5. **Curve Visualization**: Plots Train/Val Loss and Accuracy over epochs.
6. **Test Evaluation**: Computes Test Accuracy, Macro-F1, Confusion Matrix, and Per-Class Report.
7. **Export**: Saves best model weights for the MedBoard multi-agent pipeline.

**Expected training time:** ~15–20 minutes on Colab T4 GPU (or ~25–35 minutes on RTX 2050 locally).

## Cell 1 — Environment Setup (Run Every Session)

In [ ]:
import os, sys, shutil

# ── Detect environment ───────────────────────────────────────────────
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f'Running in: {"Google Colab" if IS_COLAB else "Local (VS Code / Jupyter)"}')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone or update MedBoard repo from GitHub
    if not os.path.exists('/content/MedBoard'):
        !git clone https://github.com/shivanshu0055/Medical-Image.git /content/MedBoard
    else:
        !git -C /content/MedBoard pull

    # Copy dataset from Google Drive to local Colab SSD for high-speed I/O
    if not os.path.exists('/content/data'):
        print('Copying dataset to local SSD (~30 sec)...')
        shutil.copytree('/content/drive/MyDrive/medboard/data/brisc2025', '/content/data')
        print('Dataset copied.')
    else:
        print('Dataset already available on local SSD.')

    # Install requirements
    !pip install -q -r /content/MedBoard/requirements_colab.txt

    sys.path.insert(0, '/content/MedBoard')
    DATA_ROOT   = '/content/data/classification_task'
    WEIGHTS_DIR = '/content/drive/MyDrive/medboard/weights'

else:
    # Local paths
    REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.exists(os.path.join(os.getcwd(), '..', 'modules')) else os.getcwd()
    DATA_ROOT   = os.path.join(REPO_ROOT, 'data', 'raw', 'classification_task')
    WEIGHTS_DIR = os.path.join(REPO_ROOT, 'models', 'classification')

    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)

os.makedirs(WEIGHTS_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, 'efficientnet_b0_best.pth')
FINAL_PATH      = os.path.join(WEIGHTS_DIR, 'efficientnet_b0_brisc.pth')

print('\nSetup complete!')
print(f'  Data Path    -> {DATA_ROOT}')
print(f'  Weights Path -> {WEIGHTS_DIR}')

## Cell 2 — Verify Hardware & GPU Acceleration

In [ ]:
import torch

print('=== PyTorch & Hardware Check ===')
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available:  {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU Device:      {gpu_name}')
    print(f'Total VRAM:      {vram_gb:.2f} GB')
    DEVICE_STR = 'cuda'
else:
    print('Running on CPU (training will be slower).')
    DEVICE_STR = 'cpu'

## Cell 3 — Load BRISC 2025 Classification Data

In [ ]:
from modules.clf_dataset import create_classification_dataloaders, DEFAULT_CLASSES

train_dir = os.path.join(DATA_ROOT, 'train')
test_dir  = os.path.join(DATA_ROOT, 'test')

print('Loading dataset from:', DATA_ROOT)
print('Target classes:', DEFAULT_CLASSES)

train_loader, val_loader, test_loader = create_classification_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    val_ratio=0.15,
    batch_size=32 if torch.cuda.is_available() else 8,
    image_size=224,
    num_workers=2 if IS_COLAB else 0,
    seed=42,
)

print(f'\nDataLoader Summary:')
print(f'  Train batches:      {len(train_loader)} ({len(train_loader.dataset)} images)')
print(f'  Validation batches: {len(val_loader)} ({len(val_loader.dataset)} images)')
print(f'  Test batches:       {len(test_loader)} ({len(test_loader.dataset)} images)')

# Check one batch shape
sample_images, sample_labels = next(iter(train_loader))
print(f'\nSample batch:')
print(f'  Images shape: {sample_images.shape} (B, C, H, W)')
print(f'  Labels shape: {sample_labels.shape}')

## Cell 4 — Initialize Pretrained EfficientNet-B0

In [ ]:
from modules.classifier import build_classifier, count_parameters

model = build_classifier(
    num_classes=len(DEFAULT_CLASSES),
    pretrained=True,
    dropout_rate=0.2,
)

param_info = count_parameters(model)
print('EfficientNet-B0 Built Successfully!')
print(f'  Total Parameters:     {param_info["total"]:,}')
print(f'  Trainable Parameters: {param_info["trainable"]:,}')
print(f'  Estimated Size:       ~{param_info["total"] * 4 / (1024**2):.2f} MB')

## Cell 5 — Train Classifier (AdamW + Cosine Annealing + Early Stopping)

In [ ]:
from modules.clf_trainer import train_classifier

# Hyperparameters
EPOCHS        = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 5

history = train_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    checkpoint_path=CHECKPOINT_PATH,
    device_str=DEVICE_STR,
    use_mixed_precision=True,
    class_names=DEFAULT_CLASSES,
)

## Cell 6 — Plot Training & Validation Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Loss Curve
ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs_range, history['val_loss'], 'r-s', label='Val Loss')
ax1.set_title('Cross-Entropy Loss (Lower is better)', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

# 2. Accuracy Curve
ax2.plot(epochs_range, [acc * 100 for acc in history['train_acc']], 'b-o', label='Train Acc (%)')
ax2.plot(epochs_range, [acc * 100 for acc in history['val_acc']], 'g-s', label='Val Acc (%)')
ax2.plot(epochs_range, [f1 * 100 for f1 in history['val_f1']], 'm--^', label='Val Macro-F1 (x100)')
ax2.set_title('Classification Accuracy & F1-Score', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score (%)')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
curves_path = os.path.join(WEIGHTS_DIR, 'clf_training_curves.png')
plt.savefig(curves_path, dpi=200)
plt.show()
print(f'Training curves saved to: {curves_path}')

## Cell 7 — Test Set Evaluation (Confusion Matrix & Classification Report)

In [ ]:
from modules.clf_trainer import evaluate
import torch.nn as nn
import seaborn as sns

print('Evaluating best model on un-seen Test Set (1,000 images)...')
criterion = nn.CrossEntropyLoss()
device = torch.device(DEVICE_STR)

test_metrics = evaluate(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    class_names=DEFAULT_CLASSES,
)

print('\n' + '=' * 60)
print(f'TEST ACCURACY:  {test_metrics["accuracy"] * 100:.2f}%')
print(f'TEST MACRO-F1:  {test_metrics["macro_f1"]:.4f}')
print(f'TEST LOSS:      {test_metrics["loss"]:.4f}')
print('=' * 60)

print('\nDetailed Classification Report:')
print(test_metrics['classification_report'])

# Plot Confusion Matrix
plt.figure(figsize=(7, 6))
sns.heatmap(
    test_metrics['confusion_matrix'],
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=DEFAULT_CLASSES,
    yticklabels=DEFAULT_CLASSES,
)
plt.title('Test Set Confusion Matrix', fontsize=13)
plt.xlabel('Predicted Label')
plt.ylabel('Ground Truth Label')
plt.tight_layout()

cm_path = os.path.join(WEIGHTS_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=200)
plt.show()
print(f'Confusion matrix saved to: {cm_path}')

## Cell 8 — Export Weights for MedBoard Pipeline

In [ ]:
# Copy best checkpoint to final release weights
if os.path.exists(CHECKPOINT_PATH):
    shutil.copy2(CHECKPOINT_PATH, FINAL_PATH)
    print('Saved final weights to:', FINAL_PATH)
    print(f'File size: {os.path.getsize(FINAL_PATH) / (1024**2):.2f} MB')

if IS_COLAB:
    print('\nDownload `efficientnet_b0_brisc.pth` from your Google Drive:')
    print('  MyDrive/medboard/weights/efficientnet_b0_brisc.pth')
    print('And paste it into your local project at:')
    print('  d:\\Projects\\Major\\models\\classification\\efficientnet_b0_brisc.pth')
else:
    print('\nWeights are saved directly in your local models/classification folder.')

print('\nClassification model ready for Phase 3! ✅')